# Week 55, Day 4: Regularisation paths, the elastic net and choosing λ

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w055/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: A lasso path with warm starts

Compute lam_max, then the path over 40 log-spaced values from lam_max down to lam_max / 1000, warm-starting each fit from the previous solution. Record the order in which inputs first become non-zero.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(554)

def soft_threshold(z, lam):
    return np.sign(z) * np.maximum(np.abs(z) - lam, 0)

def lasso_cd(X, y, lam, iters=100, w0=None, alpha=1.0):
    """Elastic net by coordinate descent: penalty lam * (alpha |w|_1 + (1 - alpha) |w|^2 / 2); alpha = 1 is the lasso."""
    n = len(y); w = np.zeros(X.shape[1]) if w0 is None else w0.copy(); res = y - X @ w
    for it in range(iters):
        for j in range(X.shape[1]):
            rho = X[:, j] @ (res + X[:, j] * w[j]) / n
            new = soft_threshold(rho, lam * alpha) / (1 + lam * (1 - alpha))
            res -= X[:, j] * (new - w[j]); w[j] = new
    return w

n, P = 80, 12
X = r.normal(size=(n, P)); X = (X - X.mean(0)) / X.std(0)
w_true = np.array([1.2, -0.9, 0.7, 0.5, -0.3, 0.2, 0, 0, 0, 0, 0, 0])
y = X @ w_true + 0.6 * r.normal(size=n); y = y - y.mean()

lam_max = ...
lams = lam_max * np.logspace(0, -3, 40)
path, w = np.zeros((len(lams), P)), np.zeros(P)
for k, lam in enumerate(lams):
    w = ...
    path[k] = w
entry = [np.argmax(path[:, j] != 0) if np.any(path[:, j] != 0) else len(lams) for j in range(P)]
first_in = int(np.argmin(entry))
print(f"lambda_max = {lam_max:.3f}")
print("order of entry:", [int(i) for i in np.argsort(entry)[:8]])
plt.semilogx(lams, path); plt.gca().invert_xaxis(); plt.xlabel("lambda"); plt.ylabel("coefficient"); plt.show()

In [ ]:
import numpy as np
def test_lmax():
    "lam_max zeroes every weight and anything slightly smaller does not"
    assert np.all(lasso_cd(X, y, lam_max) == 0) and np.any(lasso_cd(X, y, 0.95 * lam_max) != 0)
def test_path():
    "path has one row per lambda; the number of non-zero weights grows as lambda falls"
    nz = (path != 0).sum(1)
    assert path.shape == (len(lams), P) and nz[0] == 0 and nz[-1] > nz[len(lams) // 2]
def test_order():
    "the first input to enter is the one with the largest true effect"
    assert first_in == 0

test_lmax()
test_path()
test_order()
print("All checks passed")

### Exercise: Three copies of one signal

Build three near-copies of one signal (correlation about 0.99) plus 7 unrelated inputs; the target depends equally on the three copies. Fit the lasso (alpha = 1) and the elastic net (alpha = 0.3) at lambda = 0.1 and compare how they share weight within the group.

In [ ]:
g2 = np.random.default_rng(5541)                 # its own generator, so the result does not depend on earlier cells
base = g2.normal(size=200)
Xg = np.c_[np.stack([base + 0.1 * g2.normal(size=200) for _ in range(3)], axis=1), g2.normal(size=(200, 7))]
Xg = (Xg - Xg.mean(0)) / Xg.std(0)
yg = Xg[:, :3].sum(1) * 0.5 + 0.5 * g2.normal(size=200); yg = yg - yg.mean()
print("correlations within the group:", np.round(np.corrcoef(Xg[:, :3].T)[np.triu_indices(3, 1)], 2))
w_lasso = ...
w_enet = ...
print("lasso, group weights:      ", np.round(w_lasso[:3], 3))
print("elastic net, group weights:", np.round(w_enet[:3], 3))

In [ ]:
import numpy as np
def test_lasso():
    "the lasso concentrates the group's weight: it zeroes at least one member and its largest weight carries over 60% of the total"
    g = np.abs(w_lasso[:3]); assert np.any(g == 0) and g.max() >= 0.6 * g.sum()
def test_enet():
    "the elastic net spreads it: no weight in the group carries more than 50%"
    g = np.abs(w_enet[:3]); assert g.max() <= 0.5 * g.sum() and np.all(g > 0)

test_lasso()
test_enet()
print("All checks passed")

### Exercise: Ten-fold cross-validation along the path

For each of 10 folds, fit the whole path on the other nine folds (warm starts) and record the held-out MSE for each lambda. Find lam_min and lam_1se, and count the non-zero weights of each when refitted on all the data.

In [ ]:
folds = np.array_split(r.permutation(n), 10)
cv_err = np.zeros((10, len(lams)))
for f, te in enumerate(folds):
    tr = np.setdiff1d(np.arange(n), te)
    w = np.zeros(P)
    for k, lam in enumerate(lams):
        w = lasso_cd(X[tr], y[tr], lam, iters=30, w0=w)
        cv_err[f, k] = ...
m, se = cv_err.mean(0), cv_err.std(0) / np.sqrt(10)
k_min = np.argmin(m)
lam_min = lams[k_min]
lam_1se = ...
nz_min = int(np.sum(lasso_cd(X, y, lam_min) != 0)); nz_1se = int(np.sum(lasso_cd(X, y, lam_1se) != 0))
print(f"lambda_min {lam_min:.4f} ({nz_min} inputs); lambda_1se {lam_1se:.4f} ({nz_1se} inputs); 6 inputs truly matter")
plt.errorbar(lams, m, yerr=se, fmt="o-", ms=3); plt.xscale("log"); plt.axvline(lam_min, color="k"); plt.axvline(lam_1se, color="k", ls="--")
plt.xlabel("lambda"); plt.ylabel("10-fold CV error"); plt.show()

In [ ]:
import numpy as np
def test_shape():
    "cv_err has one row per fold and one column per lambda"
    assert cv_err.shape == (10, len(lams))
def test_rule():
    "lam_1se is the largest lambda within one SE of the minimum, and it is at least lam_min"
    m, se = cv_err.mean(0), cv_err.std(0) / np.sqrt(10)
    k = np.argmin(m); ok = m <= m[k] + se[k]
    assert lam_1se == lams[ok].max() and lam_1se >= lam_min
def test_sparser():
    "the one-SE model uses no more inputs than the minimum-error model"
    assert nz_1se <= nz_min

test_shape()
test_rule()
test_sparser()
print("All checks passed")